<a href="https://colab.research.google.com/github/chamarairesh1982/LearnPython/blob/main/Chamara_of_Week6_Training_a_Perceptron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a Perceptron

This notebook assumes you already know what a perceptron is (a single neuron, `prediction = step(x . w + b)`) and focuses entirely on **how it's trained** — the algorithm that finds good weights, applied to the same AND / OR / NOT / XOR examples from the lecture, then to a real dataset.

**By the end of this notebook you will be able to:**
1. Explain and implement the **Perceptron Learning Rule** (the training algorithm itself).
2. Train a perceptron on AND / OR / NOT and watch its decision boundary evolve epoch by epoch.
3. Explain the role of the bias term by training with and without it.
4. Demonstrate — empirically — why a single perceptron **cannot** learn XOR.
5. Train a perceptron on a **real dataset pulled from the internet**.


In [ ]:
# --- Setup ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron as SklearnPerceptron
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams["figure.figsize"] = (6, 4)
print("Libraries loaded.")

## 0. Quick Recap: The Perceptron's Forward Pass

Before we can *train* a perceptron, we need the two building blocks it's made of — the **step activation function** and the **weighted sum + bias** equation (`x . w + b`). These are defined once here and used throughout the rest of the notebook.

In [ ]:
def step_function(y, threshold=0):
    '''Perceptron's activation: 1 if y > threshold else 0.'''
    return np.where(y > threshold, 1, 0)

def weighted_sum(x, w, b):
    '''Core neuron equation: x . w + b'''
    return np.dot(x, w) + b

print("step_function and weighted_sum ready - these are the perceptron's forward pass.")

## 1. Training a Perceptron: The Learning Rule

Right now a perceptron with random weights hasn't learned anything. **Training** means *searching for the weights and bias* that make its predictions match the correct answers.

**The Perceptron Learning Rule** (Rosenblatt's algorithm):

For every training example `(x, target)`:
1. Compute the prediction: `prediction = step(x . w + b)`
2. Compute the error: `error = target - prediction`
3. Update every weight and the bias in the direction that reduces the error:

```
w = w + learning_rate * error * x
b = b + learning_rate * error
```

If the prediction is already correct, `error = 0` and **nothing changes** — the rule only nudges weights when the perceptron is wrong. Do this over the whole dataset repeatedly ("epochs") until there are zero errors (converged), or a max number of epochs is reached.

### Let's trace this by hand first, before writing any class

We'll manually step through **one weight update** so the mechanics are completely transparent before we automate it.

In [ ]:
# Manual trace of ONE perceptron learning-rule update
w = np.array([0.1, -0.1])   # arbitrary starting weights
b = 0.0                     # starting bias
lr = 0.1                    # learning rate

x_sample = np.array([1, 1]) # one training example: AND(1,1) should be 1
target = 1

# Step 1: predict
z = weighted_sum(x_sample, w, b)
prediction = step_function(z)
print(f"weighted sum = {z:.2f} -> prediction = {prediction}")

# Step 2: error
error = target - prediction
print(f"target = {target}, prediction = {prediction}, error = {error}")

# Step 3: update weights and bias (only happens because error != 0)
w_new = w + lr * error * x_sample
b_new = b + lr * error
print(f"weights: {w} -> {w_new}")
print(f"bias:    {b} -> {b_new}")
print("\nIf we'd already been correct (error=0), weights and bias would be unchanged.")

## 2. Automating It: A Perceptron Class With a Training Loop

Now let's wrap that manual update into a reusable class, and have it repeat the process across the whole dataset for multiple epochs — while **recording its history** (weights and error count per epoch) so we can visualise training happening.

In [ ]:
class Perceptron:
    '''A single artificial neuron (step activation), trained with the
    classic Perceptron Learning Rule.'''

    def __init__(self, n_inputs, learning_rate=0.1, use_bias=True):
        self.weights = np.random.randn(n_inputs) * 0.1   # small random start
        self.bias = 0.0
        self.use_bias = use_bias
        self.lr = learning_rate

    def predict(self, x):
        '''Forward pass: weighted sum (+ bias) -> step activation.'''
        z = np.dot(x, self.weights) + (self.bias if self.use_bias else 0.0)
        return step_function(z)

    def fit(self, X, y, epochs=20, verbose=False):
        '''Repeatedly apply the Perceptron Learning Rule until convergence
        or max epochs. Records weight/bias/error history for visualisation.'''
        self.weight_history = [self.weights.copy()]
        self.bias_history = [self.bias]
        self.error_history = []

        for epoch in range(epochs):
            total_errors = 0
            for xi, target in zip(X, y):
                prediction = self.predict(xi)
                error = target - prediction
                if error != 0:
                    self.weights = self.weights + self.lr * error * xi
                    if self.use_bias:
                        self.bias = self.bias + self.lr * error
                    total_errors += 1

            self.weight_history.append(self.weights.copy())
            self.bias_history.append(self.bias)
            self.error_history.append(total_errors)

            if verbose:
                print(f"Epoch {epoch+1:2d}: {total_errors} misclassifications | "
                      f"weights={np.round(self.weights,3)} | bias={self.bias:.3f}")

            if total_errors == 0:
                if verbose:
                    print(f"Converged after {epoch+1} epoch(s).")
                break
        return self.error_history

print("Perceptron class ready (with training history tracking).")

## 3. Training on AND — Watching Learning Happen

This is the **AND Gate Problem** from the slides. Rule: `x1 + x2 + b`, output TRUE only if the sum crosses the threshold. The slides state this is **linearly separable** — a single straight line can separate TRUE from FALSE. Let's verify that by actually training and plotting the boundary as it evolves.

In [ ]:
X_gates = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])   # only (1,1) -> 1, matching the slide's truth table

p_and = Perceptron(n_inputs=2, learning_rate=0.1)
# The learning rate is a number that controls how big a step the perceptron takes when it updates its weights after a mistake.
errors = p_and.fit(X_gates, y_and, epochs=20, verbose=True)

preds = np.array([p_and.predict(xi) for xi in X_gates])
print(f"\nFinal predictions: {preds} | Targets: {y_and} | Correct: {np.array_equal(preds, y_and)}")

In [ ]:
def plot_boundary(ax, weights, bias, X, y, title):
    for label, marker, color in [(0, "o", "tab:red"), (1, "s", "tab:blue")]:
        mask = y == label
        ax.scatter(X[mask, 0], X[mask, 1], marker=marker, s=150, color=color,
                   edgecolor="k", zorder=3)
    w0, w1 = weights
    x_vals = np.array([-0.5, 1.5])
    if abs(w1) > 1e-8:
        y_vals = -(w0 * x_vals + bias) / w1
        ax.plot(x_vals, y_vals, "k--", linewidth=2)
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("X1"); ax.set_ylabel("X2")
    ax.grid(alpha=0.3)

# Show the decision boundary at the START, MIDWAY, and END of training
n_snapshots = len(p_and.weight_history)
snapshot_idxs = sorted(set([0, n_snapshots // 2, n_snapshots - 1]))

fig, axes = plt.subplots(1, len(snapshot_idxs), figsize=(5*len(snapshot_idxs), 5))
if len(snapshot_idxs) == 1:
    axes = [axes]
for ax, idx in zip(axes, snapshot_idxs):
    plot_boundary(ax, p_and.weight_history[idx], p_and.bias_history[idx],
                  X_gates, y_and, f"AND gate - after epoch {idx}")
plt.tight_layout()
plt.show()

# Convergence curve
plt.plot(range(1, len(errors)+1), errors, marker="o")
plt.xlabel("Epoch"); plt.ylabel("Misclassifications")
plt.title("AND Gate: Training Errors Over Time")
plt.grid(alpha=0.3)
plt.show()

Watch the decision boundary in the snapshots above: it starts in essentially a random position, and the Perceptron Learning Rule nudges it, epoch by epoch, until it correctly separates the single blue square `(1,1)` from the three red circles. The error curve hits **zero** — this is what *convergence* looks like for a linearly separable problem.

## 4. Training on OR

Same process, different truth table — **OR Gate Problem**: output TRUE if *at least one* input is 1. Also linearly separable per the slides.

In [ ]:
y_or = np.array([0, 1, 1, 1])

p_or = Perceptron(n_inputs=2, learning_rate=0.1)
errors_or = p_or.fit(X_gates, y_or, epochs=20, verbose=True)

preds_or = np.array([p_or.predict(xi) for xi in X_gates])
print(f"\nFinal predictions: {preds_or} | Targets: {y_or} | Correct: {np.array_equal(preds_or, y_or)}")

fig, ax = plt.subplots(figsize=(5, 5))
plot_boundary(ax, p_or.weights, p_or.bias, X_gates, y_or, "OR Gate - Final Decision Boundary")
plt.show()

## 5. The Role of the Bias

The slides describe bias as a **shift** — like the intercept `c` in `y = mx + c`. *"Without bias, the decision boundary is forced through the origin; bias lets it move freely."*

Let's prove that by training two perceptrons on OR — one **with** bias (as above) and one **without** — and comparing what happens.

In [ ]:
p_or_no_bias = Perceptron(n_inputs=2, learning_rate=0.1, use_bias=False)
errors_no_bias = p_or_no_bias.fit(X_gates, y_or, epochs=20, verbose=True)

preds_no_bias = np.array([p_or_no_bias.predict(xi) for xi in X_gates])
print(f"\nNo-bias predictions: {preds_no_bias} | Targets: {y_or} | "
      f"Correct: {np.array_equal(preds_no_bias, y_or)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_boundary(axes[0], p_or.weights, p_or.bias, X_gates, y_or, "WITH bias")
plot_boundary(axes[1], p_or_no_bias.weights, 0.0, X_gates, y_or, "WITHOUT bias (forced through origin)")
plt.tight_layout()
plt.show()

Without a bias term, the decision boundary is a line **forced to pass through the origin `(0,0)`** — it can rotate, but never shift away from that point. For OR this still happens to work (the origin is on the correct side already), but for other datasets, a boundary that can't shift away from the origin may not be able to separate the classes at all. This is exactly the "bias shifts the line" idea from the slides.

## 6. Training on NOT (a single-input gate)

The **NOT Gate Problem**: one input, one output, simple sign-flip. Rule from the slides: `-x1 + b`.

In [ ]:
X_not = np.array([[0], [1]])
y_not = np.array([1, 0])

p_not = Perceptron(n_inputs=1, learning_rate=0.1)
p_not.fit(X_not, y_not, epochs=20, verbose=True)

for xi, target in zip(X_not, y_not):
    print(f"NOT({xi[0]}) -> predicted {p_not.predict(xi)}, expected {target}")

## 7. The XOR Problem — Watching Training Fail

The **XOR Gate Problem**: `(1,0)` and `(0,1)` are TRUE; `(1,1)` and `(0,0)` are FALSE. The slides state plainly: *"This can't be solved. Data literally cannot be separated with a line."*

Let's put that to the test — train exactly the same way, and watch what happens to the error curve this time.

In [ ]:
y_xor = np.array([0, 1, 1, 0])

p_xor = Perceptron(n_inputs=2, learning_rate=0.1)
errors_xor = p_xor.fit(X_gates, y_xor, epochs=50, verbose=False)

plt.plot(range(1, len(errors_xor)+1), errors_xor, marker="o", color="tab:red")
plt.xlabel("Epoch"); plt.ylabel("Misclassifications")
plt.title("XOR: Training Errors NEVER Reach Zero")
plt.grid(alpha=0.3)
plt.show()

preds_xor = np.array([p_xor.predict(xi) for xi in X_gates])
print(f"Final predictions: {preds_xor} | Targets: {y_xor}")
print(f"Accuracy: {accuracy_score(y_xor, preds_xor)*100:.0f}% "
      f"(stuck below 100%, no matter how many epochs we train)")

fig, ax = plt.subplots(figsize=(5, 5))
plot_boundary(ax, p_xor.weights, p_xor.bias, X_gates, y_xor,
              "XOR - best line the perceptron can find (still wrong)")
plt.show()

Unlike AND/OR/NOT, the error curve for XOR **oscillates and never reaches zero** — the weight update rule keeps correcting one mistake only to create another, because no straight line can separate diagonal classes. This is **"the flaw of the single-layer perceptron"** stated directly on the slides:

> *"A single perceptron can only solve linearly separable problems. AND, OR, and NOT all worked because they're linearly separable; XOR is not."*

(The lecture notes this is solved next lecture, by stacking perceptrons into a **Multi-Layer Perceptron** — outside the scope of this notebook, which is about training a single perceptron.)

## 8. Training a Perceptron on a Real Dataset

 Let's now train a perceptron on a **real dataset pulled from the internet**, and watch the same learning rule at work on real, noisy data.

**Dataset: Banknote Authentication** (UCI Machine Learning Repository) — detecting **forged banknotes** from features extracted from a Wavelet Transform of banknote images.

- `variance`, `skewness`, `curtosis`, `entropy` — numeric features from each banknote's image
- `class` — target: `0` = genuine, `1` = forged

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/banknote_authentication.csv"
column_names = ["variance", "skewness", "curtosis", "entropy", "class"]
df = pd.read_csv(url, header=None, names=column_names)

print("Shape:", df.shape)
print(df["class"].value_counts())
df.head()

In [ ]:
X = df[["variance", "skewness", "curtosis", "entropy"]].values
y = df["class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Fit the scaler on TRAINING data only - avoids data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape[0]} samples | Test: {X_test_scaled.shape[0]} samples")

We'll train **our own from-scratch Perceptron class** on this real 4-feature dataset — the exact same learning rule as Sections 1-7, just applied to real measurements instead of a 4-row truth table — and track how the error count falls across epochs.

In [ ]:
p_bank = Perceptron(n_inputs=X_train_scaled.shape[1], learning_rate=0.01)
errors_bank = p_bank.fit(X_train_scaled, y_train, epochs=50, verbose=False)  # verbose=False

plt.plot(range(1, len(errors_bank)+1), errors_bank, marker="o")
plt.xlabel("Epoch"); plt.ylabel("Misclassifications (training set)")
plt.title("Perceptron Training on Real Data: Errors Falling Over Epochs")
plt.grid(alpha=0.3)
plt.show()

train_preds = np.array([p_bank.predict(xi) for xi in X_train_scaled])
test_preds = np.array([p_bank.predict(xi) for xi in X_test_scaled])

print(f"Training accuracy: {accuracy_score(y_train, train_preds)*100:.2f}%")
print(f"Test accuracy:     {accuracy_score(y_test, test_preds)*100:.2f}%")

cm = confusion_matrix(y_test, test_preds)
ConfusionMatrixDisplay(cm, display_labels=["genuine", "forged"]).plot(cmap="Blues")
plt.title("Our From-Scratch Perceptron - Test Set")
plt.show()

It converges to a very low error rate — because, like AND and OR, this dataset is (close to) **linearly separable**. Finally, let's sanity-check our from-scratch implementation against scikit-learn's built-in, optimised `Perceptron` — they should train to a similar accuracy, since both implement the same learning rule.

In [ ]:
sk_p = SklearnPerceptron(max_iter=1000, eta0=0.01, random_state=42)  #  verbose=True
sk_p.fit(X_train_scaled, y_train)
sk_test_acc = accuracy_score(y_test, sk_p.predict(X_test_scaled))

print(f"Our from-scratch Perceptron - test accuracy : {accuracy_score(y_test, test_preds)*100:.2f}%")
print(f"scikit-learn's Perceptron  - test accuracy : {sk_test_acc*100:.2f}%")
print("\nBoth implement the same Perceptron Learning Rule, so results should be close.")

print("Learned weights:", sk_p.coef_)
print("Learned bias:   ", sk_p.intercept_)

## Exercises — Try These Yourself

1. **Learning rate:** Retrain the AND perceptron (Section 3) with `learning_rate=0.9` and `learning_rate=0.01`. Does it converge faster, slower, or does the number of epochs barely change? Why might that be, given the update rule only depends on `error * x`, not directly on how "wrong" the prediction was?
2. **Random starting weights:** Re-run Section 7 (XOR) a few times without fixing the random seed. Does the *shape* of the wrong decision boundary change each time, even though it always fails? What does that tell you about *why* it fails (a property of the data, not of the particular starting weights)?
3. **NAND gate:** Build the truth table for NAND (NOT AND) and confirm a perceptron can learn it. Plot its decision boundary.
4. **No bias on AND:** Repeat Section 5's with/without-bias comparison, but using the **AND** gate instead of OR. Does removing the bias still let it converge? Why or why not (hint: check whether the origin `(0,0)` is on the correct side of AND's decision boundary).
5. **Feature scaling:** In Section 8, retrain the from-scratch perceptron on the **unscaled** `X_train` (skip the `StandardScaler` step). Does the number of epochs to converge change? Why might unscaled features slow down (or destabilise) the weight updates?
6. **Epoch budget:** Lower `epochs=50` to `epochs=5` in Section 8. Does the perceptron have enough time to converge on the real dataset? Plot the resulting error curve and compare it to the original.